# Analisis Graf Investigasi Keamanan Siber (Distributed Memory)
Notebook ini dirancang untuk menganalisis graf memori jangka panjang (Long-term Memory) dari investigasi pelanggaran keamanan DeltaCorp menggunakan Neo4j, Cypher, dan Graph Data Science (GDS).

In [1]:
import os
from dotenv import load_dotenv
from neo4j import GraphDatabase
import pandas as pd

# Load environment variables dari file .env
load_dotenv()

NEO4J_URI = os.getenv("NEO4J_URI", "bolt://localhost:7687")
NEO4J_USERNAME = os.getenv("NEO4J_USERNAME", "neo4j")
NEO4J_PASSWORD = os.getenv("NEO4J_PASSWORD", "password")

# Inisialisasi Driver Koneksi
driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USERNAME, NEO4J_PASSWORD))
print(f"[+] Terhubung ke Neo4j pada: {NEO4J_URI}")

[+] Terhubung ke Neo4j pada: neo4j://127.0.0.1:7687


## 1. Statistik Dasar Graf
Mari kita hitung jumlah total entitas berdasarkan label / kategori (POLE+O) untuk memastikan data terimpor dengan benar.

In [2]:
def run_query(query, params=None):
    with driver.session() as session:
        result = session.run(query, params)
        return pd.DataFrame([r.data() for r in result])

# Query untuk menghitung jumlah node per label
node_count_query = """
MATCH (n)
WHERE labels(n)[0] IN ['Person', 'Object', 'Location', 'Event', 'Organization']
RETURN labels(n)[0] AS Label, count(n) AS Jumlah
ORDER BY Jumlah DESC
"""
run_query(node_count_query)

,Label,Jumlah
0,Location,421
1,Object,318
2,Person,290
3,Event,127
4,Organization,13


## 2. Analisis Centrality (Degree Centrality)
Degree Centrality mengidentifikasi entitas yang paling banyak terhubung (memiliki relasi masuk atau keluar terbanyak). Ini sangat berguna untuk mendeteksi 'Hotspot' atau entitas kunci dalam investigasi (misal: server yang diserang, atau aktor utama).

In [3]:
degree_centrality_query = """
MATCH (n)
WHERE labels(n)[0] IN ['Person', 'Object', 'Location', 'Event', 'Organization']
RETURN n.id AS Entitas, labels(n)[0] AS Tipe, count{(n)--()} AS Hubungan
ORDER BY Hubungan DESC
LIMIT 10
"""
df_centrality = run_query(degree_centrality_query)
df_centrality

,Entitas,Tipe,Hubungan
0,Sarah Connor,Person,115
1,Singapore Data Center,Location,114
2,Susan Miller,Person,110
3,Jakarta Server Room 1B,Location,107
4,Bob Johnson,Person,107
5,Jakarta Server Room 1A,Location,101
6,Singapore Server Room 3A,Location,95
7,Grace Lee,Person,95
8,Seattle Branch,Location,93
9,Singapore Server Room 3B,Location,92


## 3. Path Finding (Shortest Path)
Mencari hubungan terpendek antara entitas awal (misalnya laptop korban `Charlie Brown` atau laptop `Laptop-01`) dengan target berbahaya (`malwaredomain.com`). Ini membantu analis memahami alur infeksi (Attack Path).

In [4]:
shortest_path_query = """
MATCH p = shortestPath((start {id: 'Charlie Brown'})-[*]-(end {id: 'malwaredomain.com'}))
RETURN [node in nodes(p) | node.id] AS Jalur_Entitas, 
       [rel in relationships(p) | type(rel)] AS Alur_Relasi
"""
df_path = run_query(shortest_path_query)
if not df_path.empty:
    print("Jalur serangan yang ditemukan:")
    for i in range(len(df_path['Jalur_Entitas'][0]) - 1):
        ent1 = df_path['Jalur_Entitas'][0][i]
        rel = df_path['Alur_Relasi'][0][i]
        ent2 = df_path['Jalur_Entitas'][0][i+1]
        print(f"  ({ent1}) -[:{rel}]-> ({ent2})")
else:
    print("[-] Tidak ada jalur terpendek yang ditemukan antara Charlie Brown dan malwaredomain.com")

Jalur serangan yang ditemukan:
  (Charlie Brown) -[:BELONGS_TO]-> (Laptop-01)
  (Laptop-01) -[:COMPROMISED_BY]-> (malwaredomain.com)


## 4. Community Detection (Louvain / Weakly Connected Components)
Menggunakan Graph Data Science (GDS) plugin untuk mengelompokkan entitas ke dalam komunitas atau kluster logis berdasarkan kepadatan hubungan. Jika GDS tidak aktif, kita menyediakan fallback visualisasi relasi.

In [5]:
# Cek apakah GDS aktif
gds_check_query = "SHOW PROCEDURES YIELD name WHERE name STARTS WITH 'gds' RETURN count(name) > 0 AS gds_active"
gds_active = run_query(gds_check_query)['gds_active'][0]

if gds_active:
    print("[+] GDS Plugin Aktif. Menjalankan deteksi komunitas WCC...")
    try:
        # Proyeksikan graf
        run_query("""
        CALL gds.graph.project(
          'investigationGraph',
          ['Person', 'Object', 'Location', 'Event', 'Organization'],
          {
            ALL_RELATIONS: {
              type: '*',
              orientation: 'UNDIRECTED'
            }
          }
        )
        YIELD graphName, nodeCount, relationshipCount
        """)
        
        # Jalankan WCC
        wcc_query = """
        CALL gds.wcc.stream('investigationGraph')
        YIELD nodeId, componentId
        RETURN gds.util.asNode(nodeId).id AS Entitas, 
               labels(gds.util.asNode(nodeId))[0] AS Tipe,
               componentId AS KomunitasId
        ORDER BY KomunitasId, Entitas
        """
        df_community = run_query(wcc_query)
        
        # Drop graf proyeksi setelah selesai
        run_query("CALL gds.graph.drop('investigationGraph')")
        
        print("\n--- Hasil Komunitas (Menampilkan 15 baris pertama) ---")
        display(df_community.head(15))
        
    except Exception as e:
        print(f"[-] Error saat memproyeksikan graf GDS: {e}")
        # Drop projection jika tersisa
        try:
            run_query("CALL gds.graph.drop('investigationGraph')")
        except:
            pass
else:
    print("[-] GDS Plugin Tidak Aktif. Menampilkan komunitas berbasis struktur domain/organisasi (Fallback Cypher):")
    fallback_query = """
    MATCH (org:Organization)<-[:BELONGS_TO]-(p:Person)
    RETURN org.id AS Komunitas_Organisasi, collect(p.id) AS Anggota_Person
    """
    display(run_query(fallback_query))

[+] GDS Plugin Aktif. Menjalankan deteksi komunitas WCC...


Received notification from DBMS server: <GqlStatusObject gql_status='01N00', status_description='warn: feature deprecated. `schema` returned by the procedure `gds.graph.drop` is deprecated.', position=<SummaryInputPosition line=1, column=1, offset=0>, raw_classification='DEPRECATION', classification=<NotificationClassification.DEPRECATION: 'DEPRECATION'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'DEPRECATION', '_severity': 'WARNING', '_position': {'offset': 0, 'line': 1, 'column': 1}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: "CALL gds.graph.drop('investigationGraph')"



--- Hasil Komunitas (Menampilkan 15 baris pertama) ---


,Entitas,Tipe,KomunitasId
0,10.10.100.193,Location,0
1,10.10.101.125,Location,0
2,10.10.101.252,Location,0
3,10.10.103.103,Location,0
4,10.10.103.131,Location,0
5,10.10.103.176,Location,0
6,10.10.103.246,Location,0
7,10.10.103.64,Location,0
8,10.10.104.114,Location,0
9,10.10.105.153,Location,0


In [6]:
# Tutup koneksi driver
driver.close()
print("[+] Driver koneksi ditutup. Analisis selesai.")

[+] Driver koneksi ditutup. Analisis selesai.
